# 🧪 Semana 13-14: Bases de Datos Avanzadas y Pruebas Unitarias

**Curso:** Python, SQL, Ciencia de Datos y Análisis de Negocios  
**Duración estimada:** 2 semanas  
**Nivel:** Avanzado

---

## 📋 ¿Qué vas a aprender esta semana?

| Bloque | Temas |
|--------|-------|
| 🗄️ SQLAlchemy avanzado | Transacciones, migraciones, relaciones complejas, bulk operations |
| 🧪 unittest | Estructura de tests, assertions, setUp/tearDown, TestSuite |
| ⚡ pytest | Fixtures, parametrización, marks, plugins |
| 🔗 Tests de integración | Testear Flask + BD con pytest |
| 📊 Coverage.py | Medir y mejorar la cobertura de código |

---

> 💡 **¿Por qué testear?**  
> El código sin tests es código que no sabés si funciona.  
> Los tests te dan confianza para refactorizar, agregar funcionalidades y detectar bugs temprano.

> **Instalaciones necesarias:**
> ```bash
> pip install pytest pytest-cov sqlalchemy flask
> ```

---
## 🗄️ Bloque 1: SQLAlchemy — Transacciones y Operaciones Avanzadas

### 📘 Teoría

#### Transacciones
Una **transacción** agrupa múltiples operaciones SQL en una unidad atómica:  
o todas se ejecutan correctamente, o ninguna se aplica (rollback).

```python
with Session(engine) as session:
    try:
        session.add(objeto1)
        session.add(objeto2)
        session.commit()   # aplica todo
    except Exception:
        session.rollback() # deshace todo
        raise
```

#### Bulk Operations — inserción masiva eficiente
```python
# Lento: un INSERT por objeto
for item in miles_de_items:
    session.add(MiModelo(**item))
session.commit()

# Rápido: un solo INSERT masivo
session.bulk_insert_mappings(MiModelo, lista_de_dicts)
session.commit()

# Aún más rápido: SQL directo
session.execute(insert(MiModelo), lista_de_dicts)
session.commit()
```

#### Lazy vs Eager Loading
```python
# Lazy (default): carga relaciones solo cuando se acceden
producto = session.get(Producto, 1)
categoria = producto.categoria  # segunda query aquí

# Eager: carga todo en una sola query con JOIN
from sqlalchemy.orm import joinedload
producto = session.query(Producto).options(
    joinedload(Producto.categoria)
).get(1)  # una sola query
```

### 💡 Ejemplo resuelto 1 — Transacciones, bulk operations y eager loading

In [ ]:
from sqlalchemy import (
    create_engine, Column, Integer, String, Float,
    ForeignKey, func, insert, text
)
from sqlalchemy.orm import DeclarativeBase, relationship, Session, joinedload
import time

# ✅ Demostración de operaciones avanzadas de SQLAlchemy

class Base(DeclarativeBase): pass

class Categoria(Base):
    __tablename__ = 'categorias'
    id        = Column(Integer, primary_key=True, autoincrement=True)
    nombre    = Column(String(50), nullable=False, unique=True)
    productos = relationship('Producto', back_populates='categoria',
                             lazy='dynamic')  # query-like access

class Producto(Base):
    __tablename__ = 'productos'
    id           = Column(Integer, primary_key=True, autoincrement=True)
    nombre       = Column(String(100), nullable=False)
    precio       = Column(Float, nullable=False)
    stock        = Column(Integer, default=0)
    categoria_id = Column(Integer, ForeignKey('categorias.id'))
    categoria    = relationship('Categoria', back_populates='productos')

class MovimientoStock(Base):
    __tablename__ = 'movimientos_stock'
    id          = Column(Integer, primary_key=True, autoincrement=True)
    producto_id = Column(Integer, ForeignKey('productos.id'))
    tipo        = Column(String(10))  # 'entrada' o 'salida'
    cantidad    = Column(Integer, nullable=False)
    motivo      = Column(String(200))
    producto    = relationship('Producto')

engine = create_engine('sqlite:///:memory:', echo=False)
Base.metadata.create_all(engine)

# ── Poblar datos iniciales ──
with Session(engine) as s:
    cats = {n: Categoria(nombre=n) for n in ['Electrónica', 'Ropa', 'Hogar']}
    s.add_all(cats.values())
    s.flush()

    prods = [
        Producto(nombre='Laptop',      precio=1200.0, stock=10, categoria_id=cats['Electrónica'].id),
        Producto(nombre='Auriculares', precio=89.99,  stock=50, categoria_id=cats['Electrónica'].id),
        Producto(nombre='Camiseta',    precio=25.00,  stock=100, categoria_id=cats['Ropa'].id),
        Producto(nombre='Sillón',      precio=350.00, stock=5,  categoria_id=cats['Hogar'].id),
    ]
    s.add_all(prods)
    s.commit()

# ── TRANSACCIONES ──
print('=== Transacciones ===')

def transferir_stock(prod_origen_id, prod_destino_id, cantidad, motivo=''):
    """Transfiere stock entre productos de forma atómica."""
    with Session(engine) as s:
        try:
            origen  = s.get(Producto, prod_origen_id)
            destino = s.get(Producto, prod_destino_id)

            if origen.stock < cantidad:
                raise ValueError(f'Stock insuficiente: {origen.stock} < {cantidad}')

            # Ambas operaciones en la misma transacción
            origen.stock  -= cantidad
            destino.stock += cantidad

            # Registrar movimientos
            s.add(MovimientoStock(producto_id=prod_origen_id,  tipo='salida',  cantidad=cantidad, motivo=motivo))
            s.add(MovimientoStock(producto_id=prod_destino_id, tipo='entrada', cantidad=cantidad, motivo=motivo))

            s.commit()
            return True
        except Exception as e:
            s.rollback()
            print(f'  ❌ Rollback: {e}')
            return False

# Transferencia exitosa
ok = transferir_stock(1, 2, 3, 'rebalanceo de inventario')
print(f'  Transferencia 3 unidades Laptop→Auriculares: {"✅" if ok else "❌"}')

with Session(engine) as s:
    lap = s.get(Producto, 1)
    aur = s.get(Producto, 2)
    print(f'  Laptop stock:      {lap.stock} (era 10, ahora 7)')
    print(f'  Auriculares stock: {aur.stock} (era 50, ahora 53)')

# Transferencia fallida (stock insuficiente)
ok = transferir_stock(1, 2, 100, 'esto va a fallar')
print(f'  Transferencia 100 unidades (insuficiente): {"✅" if ok else "❌ rollback correcto"}')

# Verificar que el stock NO cambió tras el rollback
with Session(engine) as s:
    lap = s.get(Producto, 1)
    print(f'  Stock Laptop después del rollback: {lap.stock} (debe ser 7)')

# ── BULK OPERATIONS ──
print('\n=== Bulk Insert — comparación de rendimiento ===')
N = 5000

# Método 1: add() uno por uno
engine2 = create_engine('sqlite:///:memory:', echo=False)
Base.metadata.create_all(engine2)
with Session(engine2) as s:
    cat = Categoria(nombre='Test')
    s.add(cat); s.flush()
    t0 = time.perf_counter()
    for i in range(N):
        s.add(Producto(nombre=f'Prod_{i}', precio=float(i), stock=i, categoria_id=cat.id))
    s.commit()
    t_uno_por_uno = time.perf_counter() - t0

# Método 2: bulk_insert_mappings
engine3 = create_engine('sqlite:///:memory:', echo=False)
Base.metadata.create_all(engine3)
with Session(engine3) as s:
    cat = Categoria(nombre='Test')
    s.add(cat); s.flush()
    cat_id = cat.id
    t0 = time.perf_counter()
    datos = [{'nombre': f'Prod_{i}', 'precio': float(i), 'stock': i, 'categoria_id': cat_id}
             for i in range(N)]
    s.bulk_insert_mappings(Producto, datos)
    s.commit()
    t_bulk = time.perf_counter() - t0

print(f'  Insertar {N:,} registros:')
print(f'  Uno por uno:  {t_uno_por_uno*1000:.0f} ms')
print(f'  Bulk insert:  {t_bulk*1000:.0f} ms')
print(f'  Mejora:       {t_uno_por_uno/t_bulk:.1f}x más rápido')

# ── EAGER LOADING ──
print('\n=== Lazy vs Eager Loading ===')

# Lazy (N+1 queries)
with Session(engine) as s:
    t0 = time.perf_counter()
    prods = s.query(Producto).all()
    nombres_cat = [p.categoria.nombre for p in prods]  # 1 query por producto
    t_lazy = time.perf_counter() - t0

# Eager (1 sola query con JOIN)
with Session(engine) as s:
    t0 = time.perf_counter()
    prods = s.query(Producto).options(joinedload(Producto.categoria)).all()
    nombres_cat = [p.categoria.nombre for p in prods]  # sin queries adicionales
    t_eager = time.perf_counter() - t0

print(f'  Lazy loading:  {t_lazy*1000:.2f} ms ({len(prods)+1} queries)')
print(f'  Eager loading: {t_eager*1000:.2f} ms (1 query con JOIN)')
print(f'  Mejora: {t_lazy/t_eager:.1f}x')

### ✏️ Ejercicio guiado 1 — Sistema de inventario con transacciones

Implementá un sistema de inventario donde el stock se actualiza de forma transaccional.

**Pistas:**
- Una venta debe: decrementar stock del producto, registrar la venta, registrar el item de venta
- Si el stock es insuficiente, hacer rollback completo (nada se guarda)
- Usá `session.rollback()` en el bloque `except`

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import DeclarativeBase, relationship, Session

class Base2(DeclarativeBase): pass

class Prod2(Base2):
    __tablename__ = 'productos2'
    id     = Column(Integer, primary_key=True, autoincrement=True)
    nombre = Column(String(100))
    precio = Column(Float)
    stock  = Column(Integer, default=0)

class Venta(Base2):
    __tablename__ = 'ventas2'
    id    = Column(Integer, primary_key=True, autoincrement=True)
    total = Column(Float)
    items = relationship('ItemVenta2', back_populates='venta')

class ItemVenta2(Base2):
    __tablename__ = 'items_venta2'
    id          = Column(Integer, primary_key=True, autoincrement=True)
    venta_id    = Column(Integer, ForeignKey('ventas2.id'))
    producto_id = Column(Integer, ForeignKey('productos2.id'))
    cantidad    = Column(Integer)
    precio_unit = Column(Float)
    venta       = relationship('Venta', back_populates='items')

eng2 = create_engine('sqlite:///:memory:', echo=False)
Base2.metadata.create_all(eng2)

with Session(eng2) as s:
    s.add_all([
        Prod2(nombre='Laptop', precio=1200.0, stock=3),
        Prod2(nombre='Mouse',  precio=25.0,   stock=10),
    ])
    s.commit()

def procesar_venta(items_pedido):
    """
    items_pedido: lista de (producto_id, cantidad)
    Retorna (venta_id, total) o lanza ValueError si stock insuficiente.
    """
    with Session(eng2) as s:
        try:
            total = 0
            items_creados = []

            for prod_id, cantidad in items_pedido:
                prod = s.get(Prod2, prod_id)
                if not prod:
                    raise ValueError(f'Producto {prod_id} no existe')

                # ✏️ Verificá el stock y actualizalo:
                pass

            # ✏️ Creá la Venta y sus ItemVenta2:
            pass

            s.commit()
            return None  # retorná (venta.id, total)

        except Exception:
            s.rollback()
            raise

# Tests
print('=== Test venta exitosa ===')
try:
    vid, total = procesar_venta([(1, 2), (2, 3)])
    print(f'  Venta #{vid} por ${total:.2f}')
    with Session(eng2) as s:
        print(f'  Stock Laptop: {s.get(Prod2, 1).stock} (era 3)')
        print(f'  Stock Mouse:  {s.get(Prod2, 2).stock} (era 10)')
except Exception as e:
    print(f'  Error: {e}')

print('\n=== Test stock insuficiente ===')
try:
    procesar_venta([(1, 99)])
except ValueError as e:
    print(f'  ValueError capturado: {e}')
    with Session(eng2) as s:
        print(f'  Stock Laptop sin cambios: {s.get(Prod2, 1).stock}')


<details>
<summary>👀 Ver solución</summary>

```python
for prod_id, cantidad in items_pedido:
    prod = s.get(Prod2, prod_id)
    if not prod:
        raise ValueError(f'Producto {prod_id} no existe')
    if prod.stock < cantidad:
        raise ValueError(f'Stock insuficiente para {prod.nombre}: {prod.stock} < {cantidad}')
    prod.stock -= cantidad
    subtotal = prod.precio * cantidad
    total += subtotal
    items_creados.append(ItemVenta2(producto_id=prod_id, cantidad=cantidad, precio_unit=prod.precio))

venta = Venta(total=total)
s.add(venta)
s.flush()
for item in items_creados:
    item.venta_id = venta.id
    s.add(item)
return venta.id, total
```
</details>

### 🚀 Ejercicio libre 1 — Sistema bancario con transacciones

Implementá un sistema de cuentas bancarias con transferencias atómicas:

**Modelos:**
- `Cliente`: id, nombre, dni (único)
- `Cuenta`: id, cliente_id, numero (único), saldo, tipo ('caja_ahorro'/'cuenta_corriente')
- `Transaccion`: id, cuenta_origen_id, cuenta_destino_id, monto, tipo ('transferencia'/'deposito'/'extraccion'), fecha

**Operaciones (todas transaccionales):**
1. `depositar(cuenta_id, monto)` — aumentar saldo
2. `extraer(cuenta_id, monto)` — disminuir saldo (error si saldo < monto)
3. `transferir(origen_id, destino_id, monto)` — atómico: descuenta de origen, acredita en destino

**Consultas:**
1. Historial de transacciones de una cuenta
2. Clientes con saldo total (suma de todas sus cuentas) mayor a X
3. Las 3 transferencias de mayor monto del último mes

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import DeclarativeBase, relationship, Session
from datetime import datetime

# 🚀 Tu sistema bancario aquí:


---
## 🧪 Bloque 2: Pruebas Unitarias con `unittest`

### 📘 Teoría

**unittest** es el framework de testing incluido en la biblioteca estándar de Python.

#### Estructura básica
```python
import unittest

class TestMiModulo(unittest.TestCase):

    def setUp(self):
        """Se ejecuta ANTES de cada test — preparar el estado."""
        self.mi_objeto = MiClase()

    def tearDown(self):
        """Se ejecuta DESPUÉS de cada test — limpiar."""
        self.mi_objeto.cerrar()

    def test_suma_positiva(self):
        resultado = self.mi_objeto.sumar(2, 3)
        self.assertEqual(resultado, 5)

    def test_division_por_cero(self):
        with self.assertRaises(ZeroDivisionError):
            self.mi_objeto.dividir(10, 0)

if __name__ == '__main__':
    unittest.main()
```

#### Assertions más usados

| Método | Verifica |
|--------|----------|
| `assertEqual(a, b)` | `a == b` |
| `assertNotEqual(a, b)` | `a != b` |
| `assertTrue(x)` | `bool(x) is True` |
| `assertFalse(x)` | `bool(x) is False` |
| `assertIsNone(x)` | `x is None` |
| `assertIn(a, b)` | `a in b` |
| `assertRaises(Exc)` | el código lanza `Exc` |
| `assertAlmostEqual(a, b)` | floats casi iguales |
| `assertIsInstance(obj, cls)` | `isinstance(obj, cls)` |

#### Ciclo de vida de un test
```
setUpClass() → setUp() → test_1() → tearDown()
             → setUp() → test_2() → tearDown()
             → setUp() → test_3() → tearDown() → tearDownClass()
```

### 💡 Ejemplo resuelto 2 — Tests completos para una clase `Carrito`

In [ ]:
import unittest
from dataclasses import dataclass, field
from typing import Dict

# ✅ Clase a testear: carrito de compras

@dataclass
class ItemCarrito:
    nombre:   str
    precio:   float
    cantidad: int = 1

    @property
    def subtotal(self):
        return self.precio * self.cantidad

class Carrito:
    """Carrito de compras con descuentos y validaciones."""

    DESCUENTOS = {'PROMO10': 10, 'DESCUENTO20': 20, 'ESPECIAL50': 50}

    def __init__(self):
        self._items: Dict[str, ItemCarrito] = {}
        self._cupon: str = None

    def agregar(self, nombre: str, precio: float, cantidad: int = 1):
        if precio <= 0:
            raise ValueError('El precio debe ser positivo')
        if cantidad <= 0:
            raise ValueError('La cantidad debe ser positiva')
        if nombre in self._items:
            self._items[nombre].cantidad += cantidad
        else:
            self._items[nombre] = ItemCarrito(nombre, precio, cantidad)

    def eliminar(self, nombre: str):
        if nombre not in self._items:
            raise KeyError(f'Producto "{nombre}" no está en el carrito')
        del self._items[nombre]

    def aplicar_cupon(self, codigo: str):
        if codigo not in self.DESCUENTOS:
            raise ValueError(f'Cupón "{codigo}" inválido')
        self._cupon = codigo

    def subtotal(self) -> float:
        return sum(item.subtotal for item in self._items.values())

    def descuento(self) -> float:
        if not self._cupon:
            return 0.0
        pct = self.DESCUENTOS[self._cupon]
        return self.subtotal() * pct / 100

    def total(self) -> float:
        return self.subtotal() - self.descuento()

    def cantidad_items(self) -> int:
        return sum(item.cantidad for item in self._items.values())

    def esta_vacio(self) -> bool:
        return len(self._items) == 0

    def vaciar(self):
        self._items.clear()
        self._cupon = None


# ── Suite de tests completa ──
class TestCarrito(unittest.TestCase):

    def setUp(self):
        """Se ejecuta antes de CADA test — carrito fresco."""
        self.carrito = Carrito()

    # ── Tests de agregar ──
    def test_agregar_producto_simple(self):
        self.carrito.agregar('Laptop', 1200.0)
        self.assertFalse(self.carrito.esta_vacio())
        self.assertEqual(self.carrito.cantidad_items(), 1)

    def test_agregar_mismo_producto_acumula(self):
        self.carrito.agregar('Mouse', 25.0, 2)
        self.carrito.agregar('Mouse', 25.0, 3)
        self.assertEqual(self.carrito.cantidad_items(), 5)

    def test_agregar_precio_negativo_lanza_error(self):
        with self.assertRaises(ValueError):
            self.carrito.agregar('Item', -10.0)

    def test_agregar_precio_cero_lanza_error(self):
        with self.assertRaises(ValueError):
            self.carrito.agregar('Item', 0)

    def test_agregar_cantidad_negativa_lanza_error(self):
        with self.assertRaises(ValueError):
            self.carrito.agregar('Item', 10.0, -1)

    # ── Tests de eliminar ──
    def test_eliminar_producto_existente(self):
        self.carrito.agregar('Laptop', 1200.0)
        self.carrito.eliminar('Laptop')
        self.assertTrue(self.carrito.esta_vacio())

    def test_eliminar_producto_inexistente_lanza_error(self):
        with self.assertRaises(KeyError):
            self.carrito.eliminar('No existe')

    # ── Tests de totales ──
    def test_subtotal_multiple_productos(self):
        self.carrito.agregar('A', 100.0, 2)
        self.carrito.agregar('B', 50.0, 1)
        self.assertAlmostEqual(self.carrito.subtotal(), 250.0)

    def test_total_sin_cupon(self):
        self.carrito.agregar('Producto', 100.0)
        self.assertAlmostEqual(self.carrito.total(), 100.0)
        self.assertAlmostEqual(self.carrito.descuento(), 0.0)

    # ── Tests de cupones ──
    def test_cupon_valido_aplica_descuento(self):
        self.carrito.agregar('Producto', 100.0)
        self.carrito.aplicar_cupon('PROMO10')
        self.assertAlmostEqual(self.carrito.descuento(), 10.0)
        self.assertAlmostEqual(self.carrito.total(), 90.0)

    def test_cupon_20_pct(self):
        self.carrito.agregar('Producto', 200.0)
        self.carrito.aplicar_cupon('DESCUENTO20')
        self.assertAlmostEqual(self.carrito.total(), 160.0)

    def test_cupon_invalido_lanza_error(self):
        with self.assertRaises(ValueError) as ctx:
            self.carrito.aplicar_cupon('CUPONFAKE')
        self.assertIn('CUPONFAKE', str(ctx.exception))

    # ── Tests de vaciado ──
    def test_vaciar_limpia_todo(self):
        self.carrito.agregar('A', 100.0)
        self.carrito.aplicar_cupon('PROMO10')
        self.carrito.vaciar()
        self.assertTrue(self.carrito.esta_vacio())
        self.assertAlmostEqual(self.carrito.total(), 0.0)

    # ── Test de tipo de retorno ──
    def test_total_retorna_float(self):
        self.carrito.agregar('Item', 50.0)
        self.assertIsInstance(self.carrito.total(), float)


# Ejecutar los tests
loader = unittest.TestLoader()
suite  = loader.loadTestsFromTestCase(TestCarrito)
runner = unittest.TextTestRunner(verbosity=2)
resultado = runner.run(suite)

print(f'\n{'='*50}')
print(f'Tests ejecutados: {resultado.testsRun}')
print(f'Exitosos:         {resultado.testsRun - len(resultado.failures) - len(resultado.errors)}')
print(f'Fallidos:         {len(resultado.failures)}')
print(f'Errores:          {len(resultado.errors)}')

### ✏️ Ejercicio guiado 2 — Tests para la función `calcular_estadisticas`

Escribí una suite de tests para la siguiente función. Debés cubrir al menos 8 casos.

**Casos a cubrir:**
- Lista normal de números
- Lista con un solo elemento
- Lista con números negativos
- Lista con flotantes
- Lista con duplicados (para la moda)
- Lista vacía (debe lanzar `ValueError`)
- Lista con strings (debe lanzar `TypeError`)
- Verificar tipos de retorno

**Pistas:**
- `assertAlmostEqual` para flotantes (evita errores de precisión)
- `assertRaises` como context manager para verificar el mensaje de error
- `assertIsInstance` para verificar tipos

In [ ]:
import unittest

def calcular_estadisticas(datos):
    """
    Calcula media, mediana y moda de una lista de números.
    Raises:
        ValueError: si la lista está vacía
        TypeError: si contiene elementos no numéricos
    """
    if not datos:
        raise ValueError('La lista no puede estar vacía')
    if not all(isinstance(x, (int, float)) for x in datos):
        raise TypeError('Todos los elementos deben ser numéricos')

    media = sum(datos) / len(datos)

    ordenados = sorted(datos)
    n = len(ordenados)
    mediana = (ordenados[n//2-1] + ordenados[n//2]) / 2 if n % 2 == 0 else ordenados[n//2]

    frecuencias = {}
    for x in datos:
        frecuencias[x] = frecuencias.get(x, 0) + 1
    moda = max(frecuencias, key=frecuencias.get)

    return {'media': media, 'mediana': mediana, 'moda': moda}


class TestCalcularEstadisticas(unittest.TestCase):

    # ✏️ Escribí al menos 8 tests aquí:

    def test_lista_normal(self):
        resultado = calcular_estadisticas([1, 2, 3, 4, 5])
        self.assertAlmostEqual(resultado['media'], 3.0)
        # Tu código aquí

    def test_lista_vacia_lanza_error(self):
        pass  # Tu código aquí

    # ... más tests


loader = unittest.TestLoader()
suite  = loader.loadTestsFromTestCase(TestCalcularEstadisticas)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)


<details>
<summary>👀 Ver solución</summary>

```python
def test_lista_par_mediana(self):
    r = calcular_estadisticas([1, 2, 3, 4])
    self.assertAlmostEqual(r['mediana'], 2.5)

def test_lista_un_elemento(self):
    r = calcular_estadisticas([42])
    self.assertEqual(r['media'], 42)
    self.assertEqual(r['mediana'], 42)
    self.assertEqual(r['moda'], 42)

def test_negativos(self):
    r = calcular_estadisticas([-3, -1, -2])
    self.assertAlmostEqual(r['media'], -2.0)

def test_con_duplicados_moda(self):
    r = calcular_estadisticas([1, 2, 2, 2, 3])
    self.assertEqual(r['moda'], 2)

def test_lista_vacia(self):
    with self.assertRaises(ValueError):
        calcular_estadisticas([])

def test_strings_lanza_error(self):
    with self.assertRaises(TypeError):
        calcular_estadisticas([1, 'dos', 3])

def test_tipo_retorno_dict(self):
    r = calcular_estadisticas([1, 2, 3])
    self.assertIsInstance(r, dict)
    self.assertIn('media', r)
    self.assertIn('mediana', r)
    self.assertIn('moda', r)

def test_flotantes(self):
    r = calcular_estadisticas([1.5, 2.5, 3.5])
    self.assertAlmostEqual(r['media'], 2.5)
```
</details>

### 🚀 Ejercicio libre 2 — Tests para el Carrito extendido

Extendé la clase `Carrito` con estas funcionalidades y escribí tests para cada una:

1. `aplicar_descuento_por_volumen()` — si hay más de 5 unidades totales, 5% de descuento automático
2. `productos_agotados(stock_dict)` — recibe un dict `{nombre: stock}` y retorna qué productos del carrito no tienen stock suficiente
3. `exportar_json()` — retorna un JSON con todos los items, subtotales y total
4. `from_json(json_str)` — método de clase que crea un Carrito desde JSON

Para cada funcionalidad escribí al menos 3 tests que cubran: caso normal, caso borde y caso de error.

In [ ]:
import unittest
import json
from dataclasses import dataclass, field
from typing import Dict

# 🚀 Tu Carrito extendido y sus tests aquí:


---
## ⚡ Bloque 3: pytest — Testing Moderno y Eficiente

### 📘 Teoría

**pytest** es el framework de testing más popular de Python. Es más conciso y poderoso que unittest.

#### Comparación con unittest

```python
# unittest (más verboso)
class TestSuma(unittest.TestCase):
    def test_suma(self):
        self.assertEqual(sumar(2, 3), 5)

# pytest (más simple)
def test_suma():
    assert sumar(2, 3) == 5  # assert nativo de Python
```

#### Fixtures — setup reutilizable
```python
import pytest

@pytest.fixture
def carrito():
    """Fixture: crea un carrito fresco para cada test."""
    c = Carrito()
    c.agregar('Producto', 100.0)
    return c

def test_total(carrito):  # pytest inyecta el fixture automáticamente
    assert carrito.total() == 100.0
```

#### Parametrización — múltiples casos en un solo test
```python
@pytest.mark.parametrize('a,b,esperado', [
    (2, 3, 5),
    (0, 0, 0),
    (-1, 1, 0),
    (100, -50, 50),
])
def test_suma_parametrizado(a, b, esperado):
    assert sumar(a, b) == esperado
```

#### Marks — categorizar tests
```python
@pytest.mark.slow
def test_proceso_largo(): ...

@pytest.mark.skip(reason='Feature no implementada aún')
def test_pendiente(): ...

@pytest.mark.xfail(reason='Bug conocido #123')
def test_bug_conocido(): ...
```

### 💡 Ejemplo resuelto 3 — Suite completa con pytest

In [ ]:
import pytest

# ✅ Tests con pytest — fixtures, parametrización y marks

# ── Código a testear ──
class BancoCuenta:
    """Cuenta bancaria simple."""

    def __init__(self, titular: str, saldo_inicial: float = 0.0):
        if saldo_inicial < 0:
            raise ValueError('El saldo inicial no puede ser negativo')
        self.titular = titular
        self._saldo  = saldo_inicial
        self._historial = []

    @property
    def saldo(self): return self._saldo

    def depositar(self, monto: float):
        if monto <= 0:
            raise ValueError('El monto debe ser positivo')
        self._saldo += monto
        self._historial.append(('deposito', monto))

    def extraer(self, monto: float):
        if monto <= 0:
            raise ValueError('El monto debe ser positivo')
        if monto > self._saldo:
            raise ValueError(f'Saldo insuficiente: {self._saldo} < {monto}')
        self._saldo -= monto
        self._historial.append(('extraccion', monto))

    def transferir_a(self, otra: 'BancoCuenta', monto: float):
        self.extraer(monto)
        otra.depositar(monto)

    def historial(self): return list(self._historial)


# ── Fixtures ──
@pytest.fixture
def cuenta_vacia():
    return BancoCuenta('Ana García')

@pytest.fixture
def cuenta_con_saldo():
    c = BancoCuenta('Luis Pérez', saldo_inicial=1000.0)
    return c

@pytest.fixture
def dos_cuentas():
    origen  = BancoCuenta('Ana', 500.0)
    destino = BancoCuenta('Luis', 100.0)
    return origen, destino


# ── Tests básicos ──
def test_crear_cuenta_saldo_inicial(cuenta_vacia):
    assert cuenta_vacia.saldo == 0.0
    assert cuenta_vacia.titular == 'Ana García'

def test_crear_cuenta_saldo_negativo_lanza_error():
    with pytest.raises(ValueError, match='negativo'):
        BancoCuenta('Test', saldo_inicial=-100)

# ── Depósitos ──
@pytest.mark.parametrize('monto,saldo_esperado', [
    (100.0,   100.0),
    (0.01,      0.01),
    (50000.0, 50000.0),
])
def test_deposito_parametrizado(cuenta_vacia, monto, saldo_esperado):
    cuenta_vacia.depositar(monto)
    assert cuenta_vacia.saldo == pytest.approx(saldo_esperado)

def test_deposito_negativo_lanza_error(cuenta_vacia):
    with pytest.raises(ValueError):
        cuenta_vacia.depositar(-50)

def test_deposito_cero_lanza_error(cuenta_vacia):
    with pytest.raises(ValueError):
        cuenta_vacia.depositar(0)

# ── Extracciones ──
def test_extraccion_exitosa(cuenta_con_saldo):
    cuenta_con_saldo.extraer(300.0)
    assert cuenta_con_saldo.saldo == pytest.approx(700.0)

def test_extraccion_todo_el_saldo(cuenta_con_saldo):
    cuenta_con_saldo.extraer(1000.0)
    assert cuenta_con_saldo.saldo == pytest.approx(0.0)

def test_extraccion_saldo_insuficiente(cuenta_con_saldo):
    with pytest.raises(ValueError, match='insuficiente'):
        cuenta_con_saldo.extraer(9999.0)

# ── Transferencias ──
def test_transferencia_actualiza_ambas_cuentas(dos_cuentas):
    origen, destino = dos_cuentas
    origen.transferir_a(destino, 200.0)
    assert origen.saldo  == pytest.approx(300.0)
    assert destino.saldo == pytest.approx(300.0)

def test_transferencia_insuficiente_no_modifica_destino(dos_cuentas):
    origen, destino = dos_cuentas
    with pytest.raises(ValueError):
        origen.transferir_a(destino, 9999.0)
    assert destino.saldo == pytest.approx(100.0)  # sin cambios

# ── Historial ──
def test_historial_registra_operaciones(cuenta_con_saldo):
    cuenta_con_saldo.depositar(500.0)
    cuenta_con_saldo.extraer(200.0)
    h = cuenta_con_saldo.historial()
    assert len(h) == 2
    assert h[0] == ('deposito', 500.0)
    assert h[1] == ('extraccion', 200.0)

def test_historial_es_copia(cuenta_con_saldo):
    """Modificar el historial externo no afecta el interno."""
    h = cuenta_con_saldo.historial()
    h.append(('fake', 9999))
    assert len(cuenta_con_saldo.historial()) == 0


# ── Ejecutar con pytest ──
# Recolectamos los tests manualmente para ejecutar en el notebook
import inspect

tests = [
    (test_crear_cuenta_saldo_inicial,        [cuenta_vacia()]),
    (test_crear_cuenta_saldo_negativo_lanza_error, []),
    (test_deposito_negativo_lanza_error,     [cuenta_vacia()]),
    (test_extraccion_exitosa,                [cuenta_con_saldo()]),
    (test_extraccion_saldo_insuficiente,     [cuenta_con_saldo()]),
    (test_transferencia_actualiza_ambas_cuentas, [dos_cuentas()]),
    (test_historial_registra_operaciones,    [cuenta_con_saldo()]),
    (test_historial_es_copia,                [cuenta_con_saldo()]),
]

print('=== Ejecutando tests de BancoCuenta ===')
pasados = fallidos = 0
for func, args in tests:
    try:
        func(*args)
        print(f'  ✅ {func.__name__}')
        pasados += 1
    except Exception as e:
        print(f'  ❌ {func.__name__}: {e}')
        fallidos += 1

# Parametrizados manualmente
params = [(100.0, 100.0), (0.01, 0.01), (50000.0, 50000.0)]
for monto, esperado in params:
    try:
        c = cuenta_vacia()
        test_deposito_parametrizado(c, monto, esperado)
        print(f'  ✅ test_deposito_parametrizado[monto={monto}]')
        pasados += 1
    except Exception as e:
        print(f'  ❌ test_deposito_parametrizado[monto={monto}]: {e}')
        fallidos += 1

print(f'\n  Pasados: {pasados} | Fallidos: {fallidos}')

### ✏️ Ejercicio guiado 3 — Tests parametrizados

Escribí tests parametrizados para una función de validación de contraseñas.

**Reglas de la contraseña:**
- Mínimo 8 caracteres
- Al menos una mayúscula
- Al menos un número
- Al menos un carácter especial (`!@#$%^&*`)

**Pistas:**
- Usá `@pytest.mark.parametrize` con una lista de `(password, es_valida, mensaje_error)`
- Para testear múltiples casos de error, incluí el mensaje esperado en la parametrización
- Cubrí al menos 8 casos distintos

In [ ]:
import pytest
import re

def validar_password(password: str) -> tuple[bool, str]:
    """
    Valida una contraseña según las reglas del sistema.
    Retorna (es_valida, mensaje).
    """
    if len(password) < 8:
        return False, 'Mínimo 8 caracteres'
    if not any(c.isupper() for c in password):
        return False, 'Debe contener al menos una mayúscula'
    if not any(c.isdigit() for c in password):
        return False, 'Debe contener al menos un número'
    if not any(c in '!@#$%^&*' for c in password):
        return False, 'Debe contener al menos un carácter especial'
    return True, 'Contraseña válida'


# ✏️ Escribí tests parametrizados:
@pytest.mark.parametrize('password,es_valida,msg_esperado', [
    ('Abc1234!',  True,  'Contraseña válida'),
    ('corta1!A',  False, 'Mínimo'),       # menos de 8 chars
    # ✏️ Agregá más casos:
])
def test_validar_password(password, es_valida, msg_esperado):
    valida, mensaje = validar_password(password)
    assert valida == es_valida
    assert msg_esperado in mensaje


# Ejecutar manualmente:
casos = [
    ('Abc1234!',  True,  'Contraseña válida'),
    ('corta1!A',  False, 'Mínimo'),
    # ✏️ Copiá tus casos aquí:
]
print('=== Tests de validar_password ===')
for pwd, esperada, msg in casos:
    valida, mensaje = validar_password(pwd)
    ok = valida == esperada and msg in mensaje
    print(f'  {'✅' if ok else '❌'} "{pwd}" → {mensaje}')


<details>
<summary>👀 Ver solución</summary>

```python
@pytest.mark.parametrize('password,es_valida,msg_esperado', [
    ('Abc1234!',      True,  'válida'),
    ('abc1234!',      False, 'mayúscula'),    # sin mayúscula
    ('ABCD1234!',     False, 'mayúscula'),    # todo mayúsculas pero no minúsculas... wait: relee las reglas
    ('Abcdefg!',      False, 'número'),       # sin número
    ('Abcd1234',      False, 'especial'),     # sin especial
    ('Ab1!',          False, 'Mínimo'),       # muy corta
    ('',              False, 'Mínimo'),       # vacía
    ('Abc 123!',      True,  'válida'),       # espacio permitido
    ('MiPass1@2024',  True,  'válida'),       # larga y válida
])
def test_validar_password(password, es_valida, msg_esperado):
    valida, mensaje = validar_password(password)
    assert valida == es_valida
    assert msg_esperado in mensaje
```
</details>

### 🚀 Ejercicio libre 3 — Suite de tests para la app Flask

Escribí tests con pytest para la app Flask del Bloque 1 (la de Blueprints).  
Usá `app.test_client()` como fixture y cubrí:

1. **Registro:** exitoso, usuario duplicado, campos faltantes
2. **Login:** correcto, contraseña incorrecta, usuario inexistente
3. **Productos — GET:** listar todos, sin autenticación
4. **Productos — POST:** crear válido, precio negativo, sin autenticación
5. **Productos — DELETE:** eliminar propio, eliminar ajeno (403), no existente (404)
6. **Reportes:** resumen correcto, top con `n=1`

Organizá los tests en clases por recurso: `TestAuth`, `TestProductos`, `TestReportes`.

In [ ]:
import pytest
import json

# 🚀 Tu suite de tests para Flask aquí:

# Fixtures:
# @pytest.fixture
# def client():
#     app.config['TESTING'] = True
#     with app.test_client() as c:
#         yield c

# Tests:


---
## 📊 Bloque 4: Cobertura de Código con Coverage.py

### 📘 Teoría

La **cobertura de código** mide qué porcentaje de tu código fuente es ejecutado por los tests.

```bash
# Ejecutar tests con coverage
pytest --cov=mi_modulo --cov-report=term-missing tests/

# Generar reporte HTML
pytest --cov=mi_modulo --cov-report=html tests/
# Abre htmlcov/index.html en el browser
```

#### Tipos de cobertura

| Tipo | Mide |
|------|------|
| **Statement** | Líneas ejecutadas |
| **Branch** | Ramas de `if/else` (más estricto) |
| **Function** | Funciones llamadas |

#### Interpretar el reporte
```
Name              Stmts   Miss  Cover   Missing
-----------------------------------------------
carrito.py           45      8    82%   23-25, 48, 67-71
```
- **Stmts:** total de statements
- **Miss:** statements no ejecutados
- **Cover:** porcentaje cubierto
- **Missing:** líneas sin cubrir

#### ¿Cuánta cobertura es suficiente?
> - 80% es un buen punto de partida
> - 100% no garantiza que no haya bugs (podés ejecutar código sin verificar resultados)
> - Lo importante es cubrir los **caminos críticos** y los **casos borde**

### 💡 Ejemplo resuelto 4 — Medir cobertura manualmente

In [ ]:
# ✅ Simulación de análisis de cobertura
# (En la práctica usás: pytest --cov=modulo tests/)

class AnalizadorCobertura:
    """
    Simula un analizador de cobertura que rastrea qué líneas se ejecutan.
    En la práctica usarías coverage.py, que hace esto automáticamente.
    """
    def __init__(self):
        self.lineas_totales   = set()
        self.lineas_cubiertas = set()

    def registrar_linea(self, numero):
        self.lineas_totales.add(numero)
        self.lineas_cubiertas.add(numero)

    def registrar_sin_cubrir(self, numero):
        self.lineas_totales.add(numero)

    def cobertura(self):
        if not self.lineas_totales: return 0.0
        return len(self.lineas_cubiertas) / len(self.lineas_totales) * 100

    def sin_cubrir(self):
        return sorted(self.lineas_totales - self.lineas_cubiertas)


# Simular cobertura de la clase Carrito
def calcular_cobertura_carrito(tests_a_ejecutar):
    """
    Determina qué ramas del código se cubren según los tests.
    """
    cobertura = AnalizadorCobertura()

    # Líneas del método agregar() — 6 líneas lógicas
    cobertura.registrar_sin_cubrir(10)  # def agregar
    cobertura.registrar_sin_cubrir(11)  # if precio <= 0
    cobertura.registrar_sin_cubrir(12)  # raise ValueError
    cobertura.registrar_sin_cubrir(13)  # if cantidad <= 0
    cobertura.registrar_sin_cubrir(14)  # raise ValueError
    cobertura.registrar_sin_cubrir(15)  # if nombre in items
    cobertura.registrar_sin_cubrir(16)  # items[nombre].cantidad +=
    cobertura.registrar_sin_cubrir(17)  # else: items[nombre] =

    # Líneas del método aplicar_cupon() — 3 líneas
    cobertura.registrar_sin_cubrir(20)  # def aplicar_cupon
    cobertura.registrar_sin_cubrir(21)  # if codigo not in
    cobertura.registrar_sin_cubrir(22)  # raise ValueError
    cobertura.registrar_sin_cubrir(23)  # self._cupon = codigo

    # Simular qué tests cubren qué líneas
    if 'test_agregar_simple' in tests_a_ejecutar:
        cobertura.registrar_linea(10)
        cobertura.registrar_linea(11)
        cobertura.registrar_linea(13)
        cobertura.registrar_linea(15)
        cobertura.registrar_linea(17)

    if 'test_agregar_duplicado' in tests_a_ejecutar:
        cobertura.registrar_linea(15)
        cobertura.registrar_linea(16)

    if 'test_precio_negativo' in tests_a_ejecutar:
        cobertura.registrar_linea(11)
        cobertura.registrar_linea(12)

    if 'test_cantidad_negativa' in tests_a_ejecutar:
        cobertura.registrar_linea(13)
        cobertura.registrar_linea(14)

    if 'test_cupon_valido' in tests_a_ejecutar:
        cobertura.registrar_linea(20)
        cobertura.registrar_linea(21)
        cobertura.registrar_linea(23)

    if 'test_cupon_invalido' in tests_a_ejecutar:
        cobertura.registrar_linea(21)
        cobertura.registrar_linea(22)

    return cobertura

# Escenario 1: tests mínimos
tests_minimos = ['test_agregar_simple', 'test_cupon_valido']
cob1 = calcular_cobertura_carrito(tests_minimos)
print(f'Con {len(tests_minimos)} tests:')
print(f'  Cobertura: {cob1.cobertura():.1f}%')
print(f'  Líneas sin cubrir: {cob1.sin_cubrir()}')

# Escenario 2: suite completa
tests_completos = [
    'test_agregar_simple', 'test_agregar_duplicado',
    'test_precio_negativo', 'test_cantidad_negativa',
    'test_cupon_valido', 'test_cupon_invalido'
]
cob2 = calcular_cobertura_carrito(tests_completos)
print(f'\nCon {len(tests_completos)} tests:')
print(f'  Cobertura: {cob2.cobertura():.1f}%')
print(f'  Líneas sin cubrir: {cob2.sin_cubrir()}')

print('\n=== Reporte de cobertura (simulado) ===')
print('Name           Stmts   Miss  Cover   Missing')
print('-' * 50)
total = len(cob2.lineas_totales)
miss  = len(cob2.sin_cubrir())
pct   = cob2.cobertura()
sin   = ', '.join(str(l) for l in cob2.sin_cubrir()) if cob2.sin_cubrir() else 'ninguna'
print(f'carrito.py     {total:5}   {miss:4}  {pct:3.0f}%   {sin}')

print('\n💡 Para usar coverage.py real:')
print('   pip install pytest-cov')
print('   pytest --cov=carrito --cov-report=term-missing tests/')

### ✏️ Ejercicio guiado 4 — Identificar y cubrir código sin testear

Analizá la siguiente función e identificá qué ramas NO están cubiertas por los tests actuales.  
Luego escribí los tests faltantes.

**Pistas:**
- Cada rama de `if/elif/else` es una línea potencialmente sin cubrir
- Los `raise` dentro de condiciones solo se cubren si el test provoca esa condición
- Un `return` temprano también es una rama

In [ ]:
import unittest

def clasificar_transaccion(monto, tipo, saldo_actual):
    """
    Clasifica y valida una transacción bancaria.
    Retorna (aprobada: bool, mensaje: str, nuevo_saldo: float)
    """
    # Validaciones
    if not isinstance(monto, (int, float)):
        raise TypeError('El monto debe ser numérico')
    if monto <= 0:
        raise ValueError('El monto debe ser positivo')
    if tipo not in ('deposito', 'extraccion', 'transferencia'):
        raise ValueError(f'Tipo inválido: {tipo}')

    # Lógica de negocio
    if tipo == 'deposito':
        nuevo_saldo = saldo_actual + monto
        return True, f'Depósito de ${monto:.2f} aprobado', nuevo_saldo

    elif tipo in ('extraccion', 'transferencia'):
        if monto > saldo_actual:
            return False, f'Saldo insuficiente: ${saldo_actual:.2f} < ${monto:.2f}', saldo_actual
        if monto > 10000:
            return False, 'Monto supera el límite diario de $10,000', saldo_actual
        nuevo_saldo = saldo_actual - monto
        return True, f'{tipo.capitalize()} de ${monto:.2f} aprobada', nuevo_saldo

class TestClasificarTransaccion(unittest.TestCase):

    # Tests actuales (cobertura parcial)
    def test_deposito_simple(self):
        ok, msg, saldo = clasificar_transaccion(100.0, 'deposito', 500.0)
        self.assertTrue(ok)
        self.assertAlmostEqual(saldo, 600.0)

    def test_extraccion_exitosa(self):
        ok, msg, saldo = clasificar_transaccion(200.0, 'extraccion', 500.0)
        self.assertTrue(ok)
        self.assertAlmostEqual(saldo, 300.0)

    # ✏️ ¿Qué ramas faltan cubrir?
    # Escribí los tests faltantes:

    # def test_monto_string_lanza_error(self): ...
    # def test_tipo_invalido_lanza_error(self): ...
    # ... más tests


loader = unittest.TestLoader()
suite  = loader.loadTestsFromTestCase(TestClasificarTransaccion)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)


<details>
<summary>👀 Ver solución — ramas sin cubrir y tests faltantes</summary>

```python
# Ramas sin cubrir en los tests originales:
# 1. monto no numérico → TypeError
# 2. monto <= 0 → ValueError
# 3. tipo inválido → ValueError
# 4. extraccion/transferencia con saldo insuficiente
# 5. monto > 10000 (límite diario)
# 6. transferencia exitosa

def test_monto_string_lanza_error(self):
    with self.assertRaises(TypeError):
        clasificar_transaccion('cien', 'deposito', 500.0)

def test_monto_negativo_lanza_error(self):
    with self.assertRaises(ValueError):
        clasificar_transaccion(-50.0, 'deposito', 500.0)

def test_tipo_invalido_lanza_error(self):
    with self.assertRaises(ValueError, match='inválido'):
        clasificar_transaccion(100.0, 'donacion', 500.0)

def test_extraccion_saldo_insuficiente(self):
    ok, msg, saldo = clasificar_transaccion(1000.0, 'extraccion', 500.0)
    self.assertFalse(ok)
    self.assertIn('insuficiente', msg)
    self.assertAlmostEqual(saldo, 500.0)  # sin cambios

def test_limite_diario(self):
    ok, msg, saldo = clasificar_transaccion(15000.0, 'extraccion', 50000.0)
    self.assertFalse(ok)
    self.assertIn('límite', msg)

def test_transferencia_exitosa(self):
    ok, msg, saldo = clasificar_transaccion(300.0, 'transferencia', 1000.0)
    self.assertTrue(ok)
    self.assertAlmostEqual(saldo, 700.0)
```
</details>

### 🚀 Ejercicio libre 4 — Cobertura al 90%+

Tomá el código de la app Flask del Bloque 1 y:

1. Contá manualmente cuántas ramas lógicas tiene (cada `if/elif/else`, cada `raise`)
2. Identificá cuáles no están cubiertas por los tests del Ejercicio libre 3
3. Escribí los tests faltantes para llevar la cobertura teórica al 90%+
4. Documentá en una celda Markdown:
   - Total de ramas identificadas
   - Ramas cubiertas antes y después
   - Porcentaje de cobertura antes y después
   - Qué casos borde encontraste que no habías pensado antes

In [ ]:
import unittest
import json

# 🚀 Tu análisis de cobertura y tests faltantes aquí:

# 1. Ramas identificadas:
# ...

# 2. Tests faltantes:


## 📝 Análisis de cobertura

| | Antes | Después |
|---|---|---|
| Ramas totales | | |
| Ramas cubiertas | | |
| Cobertura % | | |

**Casos borde encontrados:**
- 
- 

*(Doble click para editar)*

---
## ✅ Resumen de la Semana 13-14

### Lo que aprendiste

| Tema | Conceptos clave |
|------|-----------------|
| SQLAlchemy transacciones | `commit/rollback`, atomicidad, bulk insert, lazy vs eager loading |
| unittest | `TestCase`, `setUp/tearDown`, assertions, `assertRaises` |
| pytest | Fixtures, `parametrize`, `marks`, `approx`, test de Flask |
| Coverage | Medir cobertura, identificar ramas sin cubrir, reportes |

### ➡️ Próximos pasos — Semana 15-16
- Rendimiento y escalabilidad: caché con Redis, profiling
- Seguridad: SQL injection, XSS, CSRF, encriptación
- Deployment: variables de entorno, configuración por entorno

---

### 📚 Recursos recomendados
- [pytest — Documentación oficial](https://docs.pytest.org/)
- [coverage.py — Docs](https://coverage.readthedocs.io/)
- [SQLAlchemy — Session basics](https://docs.sqlalchemy.org/en/20/orm/session_basics.html)
- [Real Python — pytest tutorial](https://realpython.com/pytest-python-testing/)
- [Test-Driven Development con Python (libro)](https://www.obeythetestinggoat.com/)